In [4]:
import pandas as pd
from pathlib import Path
score_path = Path('../../evaluation/scoreboard_eric.csv')

if score_path.exists() : 
    scoreboard = pd.read_csv(score_path)
    display(scoreboard)
else:
    print(f"{score_path} did't exist")

,rank,submission,status,reason,macro_f1,spell_f1_mean,poss,dur,n_gt_spells,gt_mean_spell_duration_s,...,n_pred_segments,fragmentation,n_graded,n_abstain_on_graded,trans_f1_12,trans_precision_12,trans_recall_12,trans_f1_25,trans_precision_25,trans_recall_25
0,1,baseline_smooth125,scored,NaN,0.581185,0.424242,16,18.460000,17,14.190588,...,17,0.809524,7500,0,0.111111,0.125000,0.10,0.166667,0.187500,0.15
1,2,baseline_smooth50,scored,NaN,0.620733,0.325581,26,10.803077,17,14.190588,...,30,1.428571,7500,0,0.081633,0.068966,0.10,0.081633,0.068966,0.10
2,3,baseline_smooth25,scored,NaN,0.619991,0.286792,36,7.580000,17,14.190588,...,40,1.904762,7500,0,0.169492,0.128205,0.25,0.203390,0.153846,0.30
3,4,eric_model,scored,NaN,0.646375,0.222222,46,6.179130,17,14.190588,...,50,2.380952,7500,0,0.144928,0.102041,0.25,0.202899,0.142857,0.35
4,5,cris_v2,scored,NaN,0.487650,0.138272,64,4.687500,17,14.190588,...,64,3.047619,7500,0,0.072289,0.047619,0.15,0.192771,0.126984,0.40
5,6,cris_v1,scored,NaN,0.524943,0.068085,77,3.802078,17,14.190588,...,82,3.904762,7500,0,0.039604,0.024691,0.10,0.118812,0.074074,0.30
6,7,baseline_raw,scored,NaN,0.597227,0.037838,131,2.019847,17,14.190588,...,137,6.523810,7500,0,0.064103,0.036765,0.25,0.141026,0.080882,0.55


In [5]:
# Daftar kolom penting sesuai urutan prioritas analisis
important_cols = [
    'rank',
    'submission',
    'status',
    'macro_f1',
    'spell_f1_mean',
    'trans_f1_12',
    'trans_f1_25',
    'fragmentation',
    'mean_abs_latency',
    'p90_abs_latency',
    'share_err',
    'n_missed',
    'n_spurious'
]

# Filter hanya kolom yang tersedia di DataFrame (aman jika ada sedikit perbedaan nama)
available_cols = [col for col in important_cols if col in scoreboard.columns]
summary_df = scoreboard[available_cols].copy()

# Urutkan berdasarkan rank terbaik (atau macro_f1 tertinggi jika rank belum terurut)
if 'rank' in summary_df.columns:
    summary_df = summary_df.sort_values(by='rank', ascending=True)
else:
    summary_df = summary_df.sort_values(by='macro_f1', ascending=False)

# Tampilkan tabel rapi dengan pembulatan 4 desimal
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(summary_df.round(4))

,rank,submission,status,macro_f1,spell_f1_mean,trans_f1_12,trans_f1_25,fragmentation,mean_abs_latency,p90_abs_latency,share_err,n_missed,n_spurious
0,1,baseline_smooth125,scored,0.5812,0.4242,0.1111,0.1667,0.8095,9.0000,18.2,0.0520,17,13
1,2,baseline_smooth50,scored,0.6207,0.3256,0.0816,0.0816,1.4286,4.0000,5.6,0.0485,18,27
2,3,baseline_smooth25,scored,0.6200,0.2868,0.1695,0.2034,1.9048,9.5000,17.5,0.0556,14,33
3,4,eric_model,scored,0.6464,0.2222,0.1449,0.2029,2.3810,11.0000,22.2,0.0115,13,42
4,5,cris_v2,scored,0.4876,0.1383,0.0723,0.1928,3.0476,13.1250,19.8,0.0106,12,55
5,6,cris_v1,scored,0.5249,0.0681,0.0396,0.1188,3.9048,14.5000,21.0,0.0017,14,75
6,7,baseline_raw,scored,0.5972,0.0378,0.0641,0.1410,6.5238,13.4545,20.0,0.0457,9,125


In [7]:
%pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)

   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   ---------------------------------------- 2/2 [jinja2]

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Kolom yang nilainya makin tinggi makin baik
higher_better = [c for c in ['macro_f1', 'spell_f1_mean', 'trans_f1_12', 'trans_f1_25'] if c in summary_df.columns]

# Kolom yang nilainya makin rendah makin baik
lower_better = [c for c in ['mean_abs_latency', 'p90_abs_latency', 'share_err', 'n_missed', 'n_spurious'] if c in summary_df.columns]

styled_table = (
    summary_df.style
    .format(precision=4)
    .highlight_max(subset=higher_better, color='#1e4620') # Hijau gelap untuk nilai tertinggi
    .highlight_min(subset=lower_better, color='#1e4620')  # Hijau gelap untuk error/latency terendah
)

display(styled_table)

,rank,submission,status,macro_f1,spell_f1_mean,trans_f1_12,trans_f1_25,fragmentation,mean_abs_latency,p90_abs_latency,share_err,n_missed,n_spurious
0,1,baseline_smooth125,scored,0.5812,0.4242,0.1111,0.1667,0.8095,9.0000,18.2000,0.0520,17,13
1,2,baseline_smooth50,scored,0.6207,0.3256,0.0816,0.0816,1.4286,4.0000,5.6000,0.0485,18,27
2,3,baseline_smooth25,scored,0.6200,0.2868,0.1695,0.2034,1.9048,9.5000,17.5000,0.0556,14,33
3,4,eric_model,scored,0.6464,0.2222,0.1449,0.2029,2.3810,11.0000,22.2000,0.0115,13,42
4,5,cris_v2,scored,0.4876,0.1383,0.0723,0.1928,3.0476,13.1250,19.8000,0.0106,12,55
5,6,cris_v1,scored,0.5249,0.0681,0.0396,0.1188,3.9048,14.5000,21.0000,0.0017,14,75
6,7,baseline_raw,scored,0.5972,0.0378,0.0641,0.1410,6.5238,13.4545,20.0000,0.0457,9,125
